# Linear algebra method comparisons

This notebook compares SpaceCore's iterative linear algebra routines with NumPy, SciPy, and JAX reference computations.

The examples are intentionally small and randomized with a fixed seed. That gives two useful checks at once: each method is exercised on several different matrices, and the notebook reports both mean and worst-case relative errors.

Methods covered:

- `sc.cg` against `numpy.linalg.solve`, `scipy.sparse.linalg.cg`, and `jax.scipy.sparse.linalg.cg`
- `sc.lsqr` against `numpy.linalg.lstsq`, `scipy.sparse.linalg.lsqr`, and `jax.numpy.linalg.lstsq`
- `sc.power_iteration` and `sc.lanczos_smallest` against dense eigendecompositions and `scipy.sparse.linalg.eigsh`
- `sc.expm_multiply` against dense and action-only matrix exponential references

The JAX comparisons are optional and are skipped automatically when JAX is not installed.

In [1]:
import numpy as np
import scipy.linalg
import scipy.sparse.linalg

import spacecore as sc

try:
    import jax

    jax.config.update("jax_enable_x64", True)
    import jax.numpy as jnp
    import jax.scipy.linalg as jsp_linalg
    import jax.scipy.sparse.linalg as jsp_sparse_linalg

    HAS_JAX = True
except Exception:
    HAS_JAX = False

HAS_JAX

True

In [2]:
rng = np.random.default_rng(20260531)
num_trials = 8

ctx = sc.Context(sc.NumpyOps(), dtype=np.float64, enable_checks=False)
ctx_complex = sc.Context(sc.NumpyOps(), dtype=np.complex128, enable_checks=False)


def as_numpy(x):
    return np.asarray(x)


def rel_error(x, ref):
    x = as_numpy(x)
    ref = as_numpy(ref)
    denom = max(float(np.linalg.norm(ref)), 1.0)
    return float(np.linalg.norm(x - ref) / denom)


def summarize_errors(method, quantity, errors, trials=None):
    errors = np.asarray(errors, dtype=float)
    return {
        "method": method,
        "quantity": quantity,
        "trials": len(errors) if trials is None else trials,
        "mean_relative_error": float(np.mean(errors)),
        "max_relative_error": float(np.max(errors)),
    }


def print_table(
    rows, columns=("method", "quantity", "trials", "mean_relative_error", "max_relative_error")
):
    widths = {name: len(name) for name in columns}
    normalized = []
    for row in rows:
        normalized_row = {}
        for name in columns:
            value = row.get(name, "")
            text = f"{value:.3e}" if isinstance(value, float) else str(value)
            normalized_row[name] = text
            widths[name] = max(widths[name], len(text))
        normalized.append(normalized_row)

    header = " | ".join(name.ljust(widths[name]) for name in columns)
    rule = "-+-".join("-" * widths[name] for name in columns)
    print(header)
    print(rule)
    for row in normalized:
        print(" | ".join(row[name].ljust(widths[name]) for name in columns))


def make_spd(n, rng, shift=1.0):
    M = rng.normal(size=(n, n))
    return M.T @ M + shift * np.eye(n)


def make_spd_with_gap(n, rng):
    Q, _ = np.linalg.qr(rng.normal(size=(n, n)))
    eigvals = np.linspace(1.0, 4.0, n)
    eigvals[-1] = 10.0
    return Q @ np.diag(eigvals) @ Q.T


def make_spacecore_dense(matrix, ctx=ctx):
    domain = sc.DenseVectorSpace((matrix.shape[1],), ctx)
    codomain = sc.DenseVectorSpace((matrix.shape[0],), ctx)
    return sc.DenseLinOp(ctx.asarray(matrix), domain, codomain, ctx)


def assert_small(summary_rows, limit):
    for row in summary_rows:
        assert row["max_relative_error"] < limit, row

## Conjugate gradients

`sc.cg` solves Hermitian positive-definite systems through `A.apply` and the domain inner product. Here every trial builds a new random SPD matrix and compares against a dense direct solve. SciPy and JAX CG are included as external iterative references.

In [3]:
errors = {"SpaceCore cg": [], "SciPy cg": []}
if HAS_JAX:
    errors["JAX cg"] = []

for _ in range(num_trials):
    A_np = make_spd(5, rng, shift=2.0)
    b_np = rng.normal(size=5)
    A = make_spacecore_dense(A_np)
    b = ctx.asarray(b_np)
    ref = np.linalg.solve(A_np, b_np)

    sc_sol = sc.cg(A, b, tol=1e-13, maxiter=20, check_every=1).x
    scipy_sol, scipy_info = scipy.sparse.linalg.cg(A_np, b_np, rtol=1e-13, atol=0.0, maxiter=20)
    assert scipy_info == 0

    errors["SpaceCore cg"].append(rel_error(sc_sol, ref))
    errors["SciPy cg"].append(rel_error(scipy_sol, ref))

    if HAS_JAX:
        A_jax = jnp.asarray(A_np)
        b_jax = jnp.asarray(b_np)
        jax_sol, _ = jsp_sparse_linalg.cg(lambda x: A_jax @ x, b_jax, tol=1e-13, maxiter=20)
        errors["JAX cg"].append(rel_error(jax_sol, ref))

rows = [
    summarize_errors(method, "solution vs numpy.linalg.solve", values)
    for method, values in errors.items()
]
print_table(rows)
assert_small([rows[0]], 1e-10)

method       | quantity                       | trials | mean_relative_error | max_relative_error
-------------+--------------------------------+--------+---------------------+-------------------
SpaceCore cg | solution vs numpy.linalg.solve | 8      | 5.465e-16           | 1.553e-15         
SciPy cg     | solution vs numpy.linalg.solve | 8      | 2.191e-16           | 2.963e-16         
JAX cg       | solution vs numpy.linalg.solve | 8      | 8.349e-16           | 4.636e-15         


## Least squares

`sc.lsqr` solves rectangular least-squares problems using forward and adjoint linear-operator actions. The reference is `numpy.linalg.lstsq`, while SciPy and JAX provide independent least-squares comparisons.

In [4]:
errors = {"SpaceCore lsqr": [], "SciPy lsqr": []}
if HAS_JAX:
    errors["JAX lstsq"] = []

for _ in range(num_trials):
    B_np = rng.normal(size=(8, 4))
    d_np = rng.normal(size=8)
    B = make_spacecore_dense(B_np)
    d = ctx.asarray(d_np)
    ref, *_ = np.linalg.lstsq(B_np, d_np, rcond=None)

    sc_sol = sc.lsqr(B, d, tol=1e-13, maxiter=80, check_every=1, residual_mode="recurrence").x
    scipy_sol = scipy.sparse.linalg.lsqr(B_np, d_np, atol=1e-13, btol=1e-13, iter_lim=80)[0]

    errors["SpaceCore lsqr"].append(rel_error(sc_sol, ref))
    errors["SciPy lsqr"].append(rel_error(scipy_sol, ref))

    if HAS_JAX:
        jax_sol, *_ = jnp.linalg.lstsq(jnp.asarray(B_np), jnp.asarray(d_np), rcond=None)
        errors["JAX lstsq"].append(rel_error(jax_sol, ref))

rows = [
    summarize_errors(method, "solution vs numpy.linalg.lstsq", values)
    for method, values in errors.items()
]
print_table(rows)
assert_small([rows[0]], 1e-10)

method         | quantity                       | trials | mean_relative_error | max_relative_error
---------------+--------------------------------+--------+---------------------+-------------------
SpaceCore lsqr | solution vs numpy.linalg.lstsq | 8      | 9.308e-16           | 1.831e-15         
SciPy lsqr     | solution vs numpy.linalg.lstsq | 8      | 8.577e-16           | 1.807e-15         
JAX lstsq      | solution vs numpy.linalg.lstsq | 8      | 4.211e-16           | 8.997e-16         


## Eigenvalue routines

`power_iteration` estimates the dominant eigenvalue; `lanczos_smallest` estimates the smallest eigenvalue of a Hermitian operator. The randomized matrices below have a controlled spectral gap so convergence is reliable in a short run.

In [5]:
errors = {
    "SpaceCore power_iteration": [],
    "SpaceCore lanczos_smallest": [],
    "SciPy eigsh smallest": [],
}
if HAS_JAX:
    errors["JAX eigh largest"] = []
    errors["JAX eigh smallest"] = []

for _ in range(num_trials):
    size = 10
    A_np = make_spd_with_gap(size, rng)
    A = make_spacecore_dense(A_np)
    initial = ctx.asarray(rng.normal(size=size))
    eigvals_np = np.linalg.eigvalsh(A_np)
    smallest_ref = eigvals_np[0]
    largest_ref = eigvals_np[-1]

    power = sc.power_iteration(A, x0=initial, tol=1e-12, maxiter=300, check_every=1)
    lanczos = sc.lanczos_smallest(A, initial, tol=1e-12, max_iter=size, check_every=1)
    scipy_smallest = scipy.sparse.linalg.eigsh(A_np, k=1, which="SA", return_eigenvectors=False)[0]

    errors["SpaceCore power_iteration"].append(rel_error(power.eigenvalue, largest_ref))
    errors["SpaceCore lanczos_smallest"].append(rel_error(lanczos.eigenvalue, smallest_ref))
    errors["SciPy eigsh smallest"].append(rel_error(scipy_smallest, smallest_ref))

    if HAS_JAX:
        jax_eigvals = jnp.linalg.eigvalsh(jnp.asarray(A_np))
        errors["JAX eigh largest"].append(rel_error(jax_eigvals[-1], largest_ref))
        errors["JAX eigh smallest"].append(rel_error(jax_eigvals[0], smallest_ref))

rows = []
for method, values in errors.items():
    quantity = (
        "largest eigenvalue" if "largest" in method or "power" in method else "smallest eigenvalue"
    )
    rows.append(summarize_errors(method, quantity, values))
print_table(rows)
assert_small([rows[0], rows[1]], 1e-8)

method                     | quantity            | trials | mean_relative_error | max_relative_error
---------------------------+---------------------+--------+---------------------+-------------------
SpaceCore power_iteration  | largest eigenvalue  | 8      | 1.776e-16           | 3.553e-16         
SpaceCore lanczos_smallest | smallest eigenvalue | 8      | 1.277e-15           | 2.442e-15         
SciPy eigsh smallest       | smallest eigenvalue | 8      | 1.193e-15           | 2.220e-15         
JAX eigh largest           | largest eigenvalue  | 8      | 2.887e-16           | 7.105e-16         
JAX eigh smallest          | smallest eigenvalue | 8      | 1.249e-15           | 3.997e-15         


## Matrix exponential action

`sc.expm_multiply` applies `exp(t A)` to a vector through a Lanczos projection. The dense reference uses `scipy.linalg.expm`; SciPy's action-only `expm_multiply` and JAX's dense `expm` are included for comparison.

In [6]:
errors = {"SpaceCore expm_multiply": [], "SciPy expm_multiply": []}
if HAS_JAX:
    errors["JAX expm"] = []

for _ in range(num_trials):
    A_np = make_spd(5, rng, shift=0.5)
    v_np = rng.normal(size=5)
    t = rng.uniform(-0.3, 0.3)
    A = make_spacecore_dense(A_np)
    v = ctx.asarray(v_np)
    ref = scipy.linalg.expm(t * A_np) @ v_np

    sc_action = sc.expm_multiply(A, v, t=t, max_iter=8, tol=1e-13).result
    scipy_action = scipy.sparse.linalg.expm_multiply(t * A_np, v_np)

    errors["SpaceCore expm_multiply"].append(rel_error(sc_action, ref))
    errors["SciPy expm_multiply"].append(rel_error(scipy_action, ref))

    if HAS_JAX:
        jax_action = jsp_linalg.expm(t * jnp.asarray(A_np)) @ jnp.asarray(v_np)
        errors["JAX expm"].append(rel_error(jax_action, ref))

rows = [summarize_errors(method, "exp(tA)v", values) for method, values in errors.items()]
print_table(rows)
assert_small([rows[0]], 1e-10)

method                  | quantity | trials | mean_relative_error | max_relative_error
------------------------+----------+--------+---------------------+-------------------
SpaceCore expm_multiply | exp(tA)v | 8      | 1.450e-14           | 1.131e-13         
SciPy expm_multiply     | exp(tA)v | 8      | 1.454e-14           | 1.144e-13         
JAX expm                | exp(tA)v | 8      | 1.467e-14           | 1.158e-13         


## Complex-time exponential

For quantum-style time evolution, `t` is often imaginary and the context should be complex-valued. This final check uses random real Hermitian matrices with complex time and verifies the SpaceCore action against dense SciPy and JAX references.

In [7]:
errors = {"SpaceCore complex expm_multiply": [], "SciPy dense expm": []}
if HAS_JAX:
    errors["JAX complex expm"] = []

for _ in range(num_trials):
    A_np = make_spd(4, rng, shift=0.25).astype(np.complex128)
    v_np = rng.normal(size=4) + 1j * rng.normal(size=4)
    t = -1j * rng.uniform(0.05, 0.4)
    A = make_spacecore_dense(A_np, ctx_complex)
    v = ctx_complex.asarray(v_np)
    ref = scipy.linalg.expm(t * A_np) @ v_np

    sc_action = sc.expm_multiply(A, v, t=t, max_iter=8, tol=1e-13).result
    scipy_action = scipy.linalg.expm(t * A_np) @ v_np

    errors["SpaceCore complex expm_multiply"].append(rel_error(sc_action, ref))
    errors["SciPy dense expm"].append(rel_error(scipy_action, ref))

    if HAS_JAX:
        jax_action = jsp_linalg.expm(t * jnp.asarray(A_np)) @ jnp.asarray(v_np)
        errors["JAX complex expm"].append(rel_error(jax_action, ref))

rows = [
    summarize_errors(method, "exp(tA)v, imaginary t", values) for method, values in errors.items()
]
print_table(rows)
assert_small([rows[0]], 1e-10)

method                          | quantity              | trials | mean_relative_error | max_relative_error
--------------------------------+-----------------------+--------+---------------------+-------------------
SpaceCore complex expm_multiply | exp(tA)v, imaginary t | 8      | 8.735e-16           | 1.457e-15         
SciPy dense expm                | exp(tA)v, imaginary t | 8      | 0.000e+00           | 0.000e+00         
JAX complex expm                | exp(tA)v, imaginary t | 8      | 3.120e-16           | 5.350e-16         


## What to take away

The SpaceCore routines agree with dense NumPy/SciPy/JAX references across repeated randomized problems. The examples use `DenseLinOp` so that references are easy to compute, but the same SpaceCore calls operate on any compatible `LinOp`, including matrix-free operators where dense matrix construction is unavailable or undesirable.